# **Health Query Chatbot Using Prompt Engineering**

**Description**

This project develops a health-related chatbot using a Large Language Model (LLM). The chatbot answers general health questions in a friendly and understandable manner while incorporating safety measures to avoid providing dangerous medical advice. Prompt engineering techniques are used to guide the model's behavior and improve response quality.



**Import Required Libraries**

**Purpose**

Import the libraries needed to connect to the language model API and interact with users.

In [1]:
!pip install transformers torch accelerate

In [2]:
from transformers import pipeline
import re

**Load the Language Model**

**Purpose**

Load an instruction-tuned language model capable of understanding and answering health-related questions.

In [4]:
chatbot = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

**Define Safety Filter**

**Purpose**

Prevent the chatbot from answering potentially harmful medical questions that require professional medical consultation.

In [5]:
dangerous_keywords = [
    "suicide",
    "kill myself",
    "overdose",
    "self harm",
    "stop medication",
    "prescription dosage",
    "emergency treatment",
    "how much medicine should i take"
]

def safety_check(user_query):

    query = user_query.lower()

    for keyword in dangerous_keywords:
        if keyword in query:
            return False

    return True

**Create Prompt Template**

**Purpose**

Use prompt engineering to instruct the model to behave as a helpful health assistant.

In [6]:
def create_prompt(user_query):

    prompt = f"""
You are a helpful medical assistant.

Guidelines:
- Provide general health information only.
- Use simple and friendly language.
- Do not diagnose diseases.
- Do not prescribe medication.
- Encourage users to consult healthcare professionals for serious concerns.
- Keep responses concise and easy to understand.

User Question:
{user_query}

Answer:
"""

    return prompt

**Generate Chatbot Response**

**Purpose**

Generate responses from the language model while enforcing safety rules.

In [7]:
def health_chatbot(user_query):

    if not safety_check(user_query):

        return (
            "I cannot provide advice on medical emergencies, "
            "medication dosages, self-harm, or other high-risk health situations. "
            "Please consult a qualified healthcare professional immediately."
        )

    prompt = create_prompt(user_query)

    response = chatbot(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )

    generated_text = response[0]["generated_text"]

    answer = generated_text.split("Answer:")[-1].strip()

    return answer

**Test Query 1**

**Purpose**

Evaluate the chatbot using a common health question.

In [8]:
question = "What causes a sore throat?"

response = health_chatbot(question)

print("Question:", question)
print()
print("Response:", response)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_th

Question: What causes a sore throat?

Response: A sore throat can be caused by several things, such as a viral infection like the common cold or flu, bacterial infections like strep throat, or environmental factors like smoke or harsh chemicals. Other causes could be allergies, acid reflux, or yelling or singing too much. If you're experiencing severe symptoms or pain lasting more than a week, it's important to consult a healthcare professional. They can provide a proper diagnosis and recommend appropriate treatments.


**Test Query 2**

**Purpose**

Evaluate the chatbot using a medication-related question.

In [9]:
question = "Is paracetamol safe for children?"

response = health_chatbot(question)

print("Question:", question)
print()
print("Response:", response)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Is paracetamol safe for children?

Response: Yes, paracetamol is generally safe for children when given at the appropriate dose. However, it's important to follow the instructions on the packaging or as advised by a healthcare professional. Overdosing can lead to serious side effects. Always consult a doctor if you have any concerns.


**Interactive Chatbot**

**Purpose**

Allow users to ask their own health-related questions.

In [10]:
while True:

    user_question = input("Ask a health question (type 'exit' to quit): ")

    if user_question.lower() == "exit":
        print("Chatbot session ended.")
        break

    response = health_chatbot(user_question)

    print("\nResponse:")
    print(response)
    print("-" * 80)

Ask a health question (type 'exit' to quit): What causes body pain


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Response:
Body pain can have many causes. It can be due to injuries, infections, inflammation, nerve damage, or chronic conditions like arthritis or fibromyalgia. Common causes include muscle strains, sprains, or overuse. Regular exercise, a balanced diet, adequate sleep, and good posture can help prevent body pain. If pain persists, it's important to consult a healthcare professional for proper diagnosis and treatment.
--------------------------------------------------------------------------------
Ask a health question (type 'exit' to quit): exit
Chatbot session ended.


**Final Insights**

**Purpose**

Summarize the project outcomes.

In [11]:
print("""
Project Summary

- Developed a health query chatbot using an LLM.
- Applied prompt engineering techniques to guide responses.
- Implemented safety filters to block harmful medical advice.
- Generated friendly and understandable health information.
- Demonstrated the use of conversational AI in healthcare support systems.
""")


Project Summary

- Developed a health query chatbot using an LLM.
- Applied prompt engineering techniques to guide responses.
- Implemented safety filters to block harmful medical advice.
- Generated friendly and understandable health information.
- Demonstrated the use of conversational AI in healthcare support systems.

